<a href="https://colab.research.google.com/github/mlbyhoang/marketing-attribution/blob/main/notebooks/04_gru_proxy_sampling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 – GRU Proxy Sampling

In [ ]:
# Load libraries
import ast
import json
import os
import random
import warnings
from collections import Counter
from statistics import mode
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from imblearn.over_sampling import ADASYN, SMOTE
from imblearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from scipy.stats import loguniform
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import (
    auc,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    brier_score_loss,
    precision_recall_curve,
    precision_score,
    recall_score, roc_auc_score, roc_curve,
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import (
    GridSearchCV, RandomizedSearchCV, StratifiedKFold, train_test_split
)
from sklearn.preprocessing import (
    LabelEncoder, MultiLabelBinarizer, OneHotEncoder, StandardScaler
)
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.layers import (
    Input,
    Embedding,
    GRU,
    Dense,
    Lambda,
    Concatenate,
    Multiply,
    Layer,
    Softmax
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import Callback
from tensorflow.keras.metrics import AUC

warnings.filterwarnings('ignore')

In [ ]:
# Verbindung mit Google Drive herstellen
from google.colab import drive, userdata
drive.mount('/content/drive')

In [ ]:
# Load filtered journey dataset
FILTERED_DATA_PATH = userdata.get("MTA_FILTERED_PATH")

df_filtered = pd.read_parquet(
    FILTERED_DATA_PATH
)

print("Shape:", df_filtered.shape)
print("Columns:", df_filtered.columns.tolist())
print(f"Conversion Rate: {df_filtered['order'].mean():.2%}")

In [ ]:
# Display first few rows
df_filtered

In [ ]:
# Touchpoint Sequenz
all_touchpoints = [tp for seq in df_filtered["sequence"] for tp in seq]

le = LabelEncoder()
le.fit(all_touchpoints)

print(f"Anzahl unique Touchpoints: {len(le.classes_)}")
print(f"Touchpoint-Typen: {le.classes_}")

X_seq_encoded = [le.transform(seq) + 1 for seq in df_filtered["sequence"]]

max_len = df_filtered["touchpoint_sum"].max()
print(f"\nmax_len: {max_len}")

X_seq_padded = np.array([
    np.pad(seq, (0, max_len - len(seq)), constant_values=0)
    for seq in X_seq_encoded
])

print(f"Shape X_seq: {X_seq_padded.shape}")

# Sicherheitscheck
print(f"\nBeispiel user1: {X_seq_padded[0]}")
print(f"Original:       {df_filtered['sequence'].iloc[0]}")


In [ ]:
# Define time features
time_feature_cols = [
    "interaction_hour_sin",
    "interaction_hour_cos",
    "interaction_weekday_sin",
    "interaction_weekday_cos",
    "tp_time_delta_min",
    "is_click"
]

n_time_features = len(time_feature_cols)

# Build time feature sequences
def build_time_sequences(df, max_len, n_time_features):
    time_seqs = []

    for _, row in df.iterrows():

        time_vals = np.array([
            list(d.values())
            for d in row["seq_times"]
        ], dtype=np.float32)

        pad_len = max_len - len(time_vals)

        if pad_len > 0:
            time_vals = np.vstack([
                time_vals,
                np.zeros(
                    (pad_len, n_time_features),
                    dtype=np.float32
                )
            ])
        else:
            time_vals = time_vals[:max_len]

        time_seqs.append(time_vals)

    return np.array(time_seqs, dtype=np.float32)


X_time = build_time_sequences(
    df_filtered,
    max_len,
    n_time_features
)

print(f"Shape X_time: {X_time.shape}")

print("\nBeispiel user1 – Zeitschritt 1:")
print(X_time[0, 0, :])

print("\nBeispiel user1 – Padding (Zeitschritt 2):")
print(X_time[0, 1, :])

In [ ]:
# Numeric journey-level features and target
X_num = (
    df_filtered[["touchpoint_sum"]]
    .values
    .astype(np.float32)
)

y = (
    df_filtered["order"]
    .values
    .reshape(-1, 1)
    .astype(np.float32)
)

vocab_size = len(le.classes_) + 1
n_num_features = X_num.shape[1]

print(f"vocab_size:      {vocab_size}")
print(f"n_time_features: {n_time_features}")
print(f"n_num_features:  {n_num_features}")
print(f"max_len:         {max_len}")

In [ ]:
# Train / validation / test split
indices = np.arange(len(df_filtered))

idx_trainval, idx_test = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=y.ravel()
)

idx_train, idx_val = train_test_split(
    idx_trainval,
    test_size=0.125,
    random_state=42,
    stratify=y[idx_trainval].ravel()
)

# Sequence input
X_seq_train, X_seq_val, X_seq_test = (
    X_seq_padded[idx_train],
    X_seq_padded[idx_val],
    X_seq_padded[idx_test]
)

# Time-feature input
X_time_train, X_time_val, X_time_test = (
    X_time[idx_train],
    X_time[idx_val],
    X_time[idx_test]
)

# Numerical journey-level input
X_num_train, X_num_val, X_num_test = (
    X_num[idx_train],
    X_num[idx_val],
    X_num[idx_test]
)

# Target
y_train, y_val, y_test = (
    y[idx_train],
    y[idx_val],
    y[idx_test]
)

print(f"Train: {len(y_train)}")
print(f"Val:   {len(y_val)}")
print(f"Test:  {len(y_test)}")

print(f"\nCR Train: {y_train.mean():.2%}")
print(f"CR Val:   {y_val.mean():.2%}")
print(f"CR Test:  {y_test.mean():.2%}")

In [ ]:
# Scale numerical features using training data only
scaler_num = StandardScaler()

X_num_train = scaler_num.fit_transform(X_num_train)
X_num_val = scaler_num.transform(X_num_val)
X_num_test = scaler_num.transform(X_num_test)

In [ ]:
# Standard hyperparameters for the GRU proxy model
embedding_dim = 16
gru_units = 32
dropout = 0.2
learning_rate = 1e-3
dense_units = 16

print("===== Standard Hyperparameters =====")
print(f"embedding_dim: {embedding_dim}")
print(f"gru_units:     {gru_units}")
print(f"dropout:       {dropout}")
print(f"learning_rate: {learning_rate}")
print(f"dense_units:   {dense_units}")

In [ ]:
# Random oversampling for multi-input sequence data
def oversample_sequences(
    X_seq,
    X_time,
    X_num,
    y,
    sampling_strategy
):
    idx = np.arange(len(y)).reshape(-1, 1)

    ros = RandomOverSampler(
        sampling_strategy=sampling_strategy,
        random_state=42
    )

    idx_resampled, y_resampled = ros.fit_resample(
        idx,
        y.ravel()
    )

    idx_resampled = idx_resampled.ravel()

    return (
        X_seq[idx_resampled],
        X_time[idx_resampled],
        X_num[idx_resampled],
        y_resampled.reshape(-1, 1).astype(np.float32)
    )

In [ ]:
# Proxy-Modell
def build_proxy_model():

    tf.keras.backend.clear_session()

    seq_input = Input(shape=(max_len,), name="seq_input")
    time_input = Input(shape=(max_len, n_time_features), name="time_input")
    num_input = Input(shape=(n_num_features,), name="num_input")

    embedding = Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        mask_zero=True
    )(seq_input)

    num_repeated = Lambda(
        lambda x: tf.repeat(
            tf.expand_dims(x, axis=1),
            repeats=max_len,
            axis=1
        )
    )(num_input)

    x = Concatenate(axis=-1)([embedding, time_input, num_repeated])

    x = GRU(
        gru_units,
        return_sequences=False,   # ✅ einzelnes GRU-Layer, direkt auf finalen Hidden State
        dropout=dropout,
        recurrent_dropout=dropout / 2
    )(x)

    x = Dense(dense_units, activation="relu")(x)

    output = Dense(1, activation="sigmoid", name="conversion")(x)

    model = Model(
        inputs=[seq_input, time_input, num_input],
        outputs=output
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[
            AUC(name="roc_auc"),
            AUC(name="pr_auc", curve="PR")
        ]
    )

    return model

In [ ]:
# Generate imbalance strategy candidates
search_space = {
    "class_weight": [None, 5.0, 7.0, 10.0, 13.0],
    "sampling_strategy": [None, 0.3, 0.5, 1.0]
}

random.seed(42)

candidates = []
seen = set()

while len(candidates) < 10:
    cw = random.choice(search_space["class_weight"])
    strategy = random.choice(search_space["sampling_strategy"])

    key = (cw, strategy)

    if key not in seen:
        seen.add(key)

        candidates.append({
            "class_weight": cw,
            "sampling_strategy": strategy
        })

print(f"Anzahl Kandidaten: {len(candidates)}")

print("\n===== Kandidaten =====")
for i, candidate in enumerate(candidates, start=1):
    print(
        f"Kandidat {i:2d}: "
        f"class_weight={candidate['class_weight']}, "
        f"sampling={candidate['sampling_strategy']}"
    )

In [ ]:
# Random Search Training
save_path = '/content/drive/MyDrive/Multi Touch Attribution/notebook_ma/results/'
os.makedirs(save_path, exist_ok=True)

results_rs = []
best_pr_auc = 0
best_cw = None
best_strategy = None

for i, candidate in enumerate(candidates):
    cw       = candidate['class_weight']
    strategy = candidate['sampling_strategy']

    print(f"\n{'='*50}")
    print(f"Kandidat {i+1}/{len(candidates)}")
    print(f"class_weight={cw}, sampling_strategy={strategy}")
    print(f"{'='*50}")

    if strategy is not None:
        X_seq_tr, X_time_tr, X_num_tr, y_tr = oversample_sequences(
            X_seq_train, X_time_train, X_num_train, y_train,
            sampling_strategy=strategy
        )
        print(f"Samples: {len(y_tr)}, CR: {y_tr.mean():.2%}")
    else:
        X_seq_tr  = X_seq_train
        X_time_tr = X_time_train
        X_num_tr  = X_num_train
        y_tr      = y_train
        print(f"Samples: {len(y_tr)}, CR: {y_tr.mean():.2%}")

    cw_dict = {0: 1.0, 1: cw} if cw is not None else None

    model_rs = build_proxy_model()

    model_rs.fit(
        [X_seq_tr, X_time_tr, X_num_tr],
        y_tr,
        validation_data=(
            [X_seq_val, X_time_val, X_num_val],   # ✅ Val statt Test
            y_val
        ),
        epochs=20,
        batch_size=64,
        class_weight=cw_dict,
        callbacks=[
            EarlyStopping(
                monitor='val_pr_auc',
                patience=3,
                mode='max',
                restore_best_weights=True
            ),
            ModelCheckpoint(
                filepath=f'{save_path}proxy_kandidat_{i+1}.keras',
                monitor='val_pr_auc',
                save_best_only=True,
                mode='max',
                verbose=0
            )
        ],
        verbose=1
    )

    y_prob = model_rs.predict(
        [X_seq_val, X_time_val, X_num_val],
        verbose=0
    ).flatten()

    pr_auc  = average_precision_score(y_val.flatten(), y_prob)
    roc_auc = roc_auc_score(y_val.flatten(), y_prob)
    ll      = log_loss(y_val.flatten(), y_prob)

    print(f"\nPR-AUC (Val):   {pr_auc:.4f}")
    print(f"ROC-AUC (Val):  {roc_auc:.4f}")
    print(f"Log Loss (Val): {ll:.4f}")

    if pr_auc > best_pr_auc:
        best_pr_auc = pr_auc
        best_cw = cw
        best_strategy = strategy
        model_rs.save(f'{save_path}proxy_random_search_best.keras')
        print(f"✅ Neues bestes Modell gespeichert (PR-AUC={pr_auc:.4f})")

    results_rs.append({
        'Kandidat':          i+1,
        'class_weight':      cw,
        'sampling_strategy': strategy,
        'PR-AUC':            round(pr_auc, 4),
        'ROC-AUC':           round(roc_auc, 4),
        'Log Loss':          round(ll, 4)
    })

    pd.DataFrame(results_rs).to_csv(
        f'{save_path}proxy_rs_checkpoint.csv',
        index=False
    )
    print(f"✅ Checkpoint gespeichert")

rs_df = pd.DataFrame(results_rs)

print("\n===== Random Search Ergebnisse (auf Validierungsdaten) =====")
print(rs_df.sort_values('PR-AUC', ascending=False).to_string(index=False))

print(f"\n===== Beste Kombination (basierend auf Val-PR-AUC) =====")
print(f"class_weight:      {best_cw}")
print(f"sampling_strategy: {best_strategy}")
print(f"PR-AUC (Val):      {best_pr_auc:.4f}")

rs_df.to_csv(f'{save_path}proxy_rs_final.csv', index=False)

print(f"\n✅ Alles gespeichert:")
print(f"→ Ergebnisse:  proxy_rs_final.csv")
print(f"→ Proxy-Modell (Zwischenprodukt aus Suche, NICHT final): proxy_random_search_best.keras")